In [1]:
"""
Copyright (c) Facebook, Inc. and its affiliates.

This source code is licensed under the MIT license found in the
LICENSE file in the root directory of this source tree.
"""

import bisect
import pickle
from pathlib import Path

import lmdb
import numpy as np
from torch.utils.data import Dataset
# from torch_geometric.data import Batch


class LmdbDataset(Dataset):
    r"""Dataset class to load from LMDB files containing relaxation
    trajectories or single point computations.

    Useful for Structure to Energy & Force (S2EF), Initial State to
    Relaxed State (IS2RS), and Initial State to Relaxed Energy (IS2RE) tasks.

    Args:
            config (dict): Dataset configuration
            transform (callable, optional): Data transform function.
                    (default: :obj:`None`)
    """

    def __init__(self, src, transform=None, **kwargs):
        super(LmdbDataset, self).__init__()

        self.path = Path(src)
        if not self.path.is_file():
            db_paths = sorted(self.path.glob("*.lmdb"))
            assert len(db_paths) > 0, f"No LMDBs found in '{self.path}'"

            self.metadata_path = self.path / "metadata.npz"

            self._keys, self.envs = [], []
            for db_path in db_paths:
                self.envs.append(self.connect_db(db_path))
                length = pickle.loads(
                    self.envs[-1].begin().get("length".encode("ascii"))
                )
                self._keys.append(list(range(length)))

            keylens = [len(k) for k in self._keys]
            self._keylen_cumulative = np.cumsum(keylens).tolist()
            self.num_samples = sum(keylens)
        else:
            self.metadata_path = self.path.parent / "metadata.npz"
            self.env = self.connect_db(self.path)
            try:
                # Try to get the stored length value first
                self.num_samples = pickle.loads(
                    self.env.begin().get("length".encode("ascii"))
                )
            except (TypeError, KeyError):
                # Fallback to entries count if length key doesn't exist
                self.num_samples = self.env.stat()["entries"]
            
            self._keys = [
                f"{j}".encode("ascii")
                for j in range(self.num_samples)
            ]

        self.transform = transform

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        if idx >= self.num_samples:
            raise IndexError(f"Index {idx} out of range for dataset with {self.num_samples} samples")
        
        if not self.path.is_file():
            # Figure out which db this should be indexed from.
            db_idx = bisect.bisect(self._keylen_cumulative, idx)
            # Extract index of element within that db.
            el_idx = idx
            if db_idx != 0:
                el_idx = idx - self._keylen_cumulative[db_idx - 1]
            assert el_idx >= 0

            # Return features.
            datapoint_pickled = (
                self.envs[db_idx]
                .begin()
                .get(f"{self._keys[db_idx][el_idx]}".encode("ascii"))
            )
            data_object = pickle.loads(datapoint_pickled)
            data_object.id = f"{db_idx}_{el_idx}"
        else:
            datapoint_pickled = self.env.begin().get(self._keys[idx])
            if datapoint_pickled is None:
                raise KeyError(f"No data found for index {idx}")
            data_object = pickle.loads(datapoint_pickled)

        if self.transform is not None:
            data_object = self.transform(data_object)

        return data_object

    def connect_db(self, lmdb_path=None):
        env = lmdb.open(
            str(lmdb_path),
            subdir=False,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            max_readers=1,
            map_size=1099511627776 * 2,
        )
        return env

    def close_db(self):
        if not self.path.is_file():
            for env in self.envs:
                env.close()
        else:
            self.env.close()

In [2]:
lmdb_path = 'archive/ts1x-val.lmdb'
lmdb_dataset = LmdbDataset(lmdb_path)
print(len(lmdb_dataset))
print(lmdb_dataset[0])

50844
Data(pos=[16, 3], rxn=0, energy=-7390.234375, ae=0.744897186756134, forces=[16, 3], charges=[16], one_hot=[16, 5], natoms=16, hessian=[2304])


In [3]:
print(lmdb_dataset[0].pos)


tensor([[-1.1111,  0.3288, -0.8598],
        [-0.1311, -0.5483, -0.0901],
        [-0.5770, -1.8858,  0.3172],
        [ 0.3138, -1.7363, -0.7694],
        [ 1.0656,  0.0653,  0.5665],
        [ 1.7488,  1.0493, -0.0062],
        [-0.8670,  1.3917, -0.7975],
        [-2.1378,  0.2127, -0.4928],
        [-1.0888,  0.0246, -1.9077],
        [-0.9002,  1.6226,  1.7372],
        [-1.5674, -2.2162,  0.0043],
        [-0.1914, -2.3622,  1.2193],
        [ 1.4198, -0.4135,  1.4792],
        [ 1.3958,  1.5092, -0.9276],
        [ 2.6730,  1.4291,  0.4149],
        [-0.2113,  1.8024,  1.5248]])


In [4]:
print(lmdb_dataset[0].forces)


tensor([[ 5.8987e-02, -2.2219e-02, -8.0368e-02],
        [ 6.5463e-02, -8.0844e-02, -1.1502e-01],
        [ 5.1287e-05, -4.9565e-02, -8.9776e-03],
        [-1.4010e-01, -1.3698e-01, -1.4475e-01],
        [ 5.2313e-02, -4.6030e-02, -9.4228e-02],
        [ 3.7312e-02,  2.0158e-02,  3.1062e-02],
        [ 7.6492e-02, -2.5249e-02, -1.8071e-02],
        [ 9.3675e-02, -4.7758e-02,  2.1797e-02],
        [-5.0335e-02,  2.0088e-02, -1.2498e-01],
        [-1.7475e-01,  1.8376e-01,  1.6052e-01],
        [-1.4447e-01,  6.1441e-02,  2.2375e-01],
        [ 1.2372e-01, -2.6316e-02, -2.1026e-02],
        [ 4.1745e-02,  2.4941e-02, -3.1844e-02],
        [ 8.5798e-02,  3.8159e-02,  1.4910e-02],
        [ 5.4043e-02,  2.9481e-02,  2.8016e-02],
        [-1.7808e-01,  5.8226e-02,  1.5961e-01]])


In [7]:
print(len(lmdb_dataset[0].hessian))
print(lmdb_dataset[0].hessian)


2304
tensor([53.5074,  2.9666, -2.6815,  ..., -9.8805, -2.4071,  3.5190])


2304

In [ ]:
lmdb_path = 'archive/RGD1.lmdb'
lmdb_dataset = LmdbDataset(lmdb_path)
print(len(lmdb_dataset))
print(lmdb_dataset[0])


60000
Data(pos=[22, 3], energy=-11002.2392578125, forces=[22, 3], charges=[22], one_hot=[22, 5], natoms=22, hessian=[4356], freq=[66], eig_values=[66], force_constant=[66])


In [9]:
print(lmdb_dataset[0].charges)
print(lmdb_dataset[0].one_hot)

tensor([6, 6, 6, 7, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 8, 1, 1, 1, 1],
       dtype=torch.int32)
tensor([[0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.]])


In [ ]:
# Derive atomic numbers from one_hot or charges for sample 0
sample = lmdb_dataset[0]
import torch
if hasattr(sample, 'one_hot') and sample.one_hot is not None:
    types = torch.as_tensor(sample.one_hot).argmax(dim=-1).long()
    # Provide your one-hot to Z map here; example: H=1, C=6, N=7, O=8, S=16
    type_map = [1, 6, 7, 8, 16]
    z = torch.tensor([type_map[i] for i in types.tolist()], dtype=torch.long)
    print('atomic_numbers from one_hot ->', z)
elif hasattr(sample, 'charges') and sample.charges is not None:
    z = torch.as_tensor(sample.charges).round().clamp(1, 118).long()
    print('atomic_numbers from charges ->', z)
else:
    print('No one_hot or charges found; cannot derive species here.')

In [ ]:
# Hessian reshape and symmetry check on sample 0
import torch
import math
sample = lmdb_dataset[0]
H = torch.as_tensor(sample.hessian)
N = sample.pos.shape[0] if hasattr(sample, 'pos') else int(math.sqrt(H.numel()//9))
D = N * 3
# Accept flat or (D*D) or already (D,D)
if H.ndim == 1 and H.numel() == D*D:
    H = H.view(D, D)
elif H.ndim == 2 and H.shape == (D, D):
    pass
else:
    print('Unexpected Hessian shape:', tuple(H.shape))
print('Hessian shape (D,D):', H.shape)
sym_err = (H - H.T).abs().max().item()
print('Max asymmetry |H-H^T|_max =', sym_err)